# ═══════════════════════════════════════════════════════════════════════════════
# PS2 — Cascade Chain Analysis
# SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor
#
# Architecture : CUBIC MARS AWS v6
# Gold table   : mars_dev.gold.device_ps2_chains
# Grain        : one row per (device_id, transit_day) where chain_length >= 2
#                — cascade days only; non-cascade days are NOT in this table
#
# Primary target   : has_cash_cascade  (binary — BHU+CHU co-failure, most costly)
# Secondary targets: has_gate_mech_in_chain | has_printer_in_chain | chain_length
#
# Models       : XGBoost + LightGBM + Markov Chain + HMM(3-state) + Association Rules
#                + Stacking Ensemble
# Feasibility  : 85%  (15% gap: SVN_STAGE.CMDB_CI / CMDB_REL = 0 rows)
# ═══════════════════════════════════════════════════════════════════════════════

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "deltalake>=0.17","s3fs","pyarrow",
    "scikit-learn>=1.4","xgboost>=2.0","lightgbm>=4.3","catboost>=1.2",
    "hmmlearn>=0.3","mlxtend>=0.23",
    "optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0","matplotlib","seaborn","networkx",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import shap, optuna, boto3, sagemaker, joblib, tarfile

from deltalake                       import DeltaTable
from sklearn.model_selection         import TimeSeriesSplit
from sklearn.preprocessing           import StandardScaler, LabelEncoder
from sklearn.pipeline                import Pipeline
from sklearn.linear_model            import LogisticRegression
from sklearn.ensemble                import (RandomForestClassifier,
                                             StackingClassifier)
from sklearn.calibration             import CalibratedClassifierCV
from sklearn.metrics                 import (roc_auc_score, f1_score,
                                             average_precision_score,
                                             classification_report,
                                             confusion_matrix)
from xgboost   import XGBClassifier
from lightgbm  import LGBMClassifier, early_stopping, log_evaluation
from catboost  import CatBoostClassifier
from hmmlearn  import hmm
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing     import TransactionEncoder

import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps2_chains"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3",        region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps2-cascade-analysis"
MLFLOW_MODEL_NAME   = "ps2-cascade-analysis"
MODEL_REGISTRY_NAME = "chicago-ps2-cascade-analysis"
FS_GROUP_NAME       = "chicago-ps2-features"
ENDPOINT_NAME       = "chicago-ps2-cascade-v1"

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3 : {GOLD_S3}")

In [ ]:
# ── CELL 4 : Load Gold (Delta) ────────────────────────────────────────────────
dt = DeltaTable(GOLD_S3)
df = dt.to_pandas()
GOLD_VERSION = dt.version()
df["transit_day"] = pd.to_datetime(df["transit_day"])

print(f"Shape        : {df.shape}")
print(f"Delta version: {GOLD_VERSION}")
print(f"\nCascade type distribution (% of all cascade days):")
for col in ["has_cash_cascade","has_gate_mech_in_chain","has_printer_in_chain",
            "has_csc_reader_in_chain","has_dopp_in_chain","has_scrst_in_chain"]:
    if col in df.columns:
        print(f"  {col:<30}: {df[col].mean():.2%}")
print(f"\nChain length stats:\n{df['chain_length'].describe().round(2)}")

In [ ]:
# ── CELL 5 : EDA ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Cascade type prevalence
type_cols = [c for c in ["has_cash_cascade","has_gate_mech_in_chain",
                          "has_printer_in_chain","has_csc_reader_in_chain",
                          "has_dopp_in_chain","has_scrst_in_chain"]
             if c in df.columns]
type_rates = df[type_cols].mean().sort_values(ascending=False)
type_rates.plot(kind="bar", ax=axes[0], color="#1565C0", edgecolor="white")
axes[0].set_title("Cascade type prevalence (% of cascade days)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# Chain length distribution
df["chain_length"].plot(kind="hist", ax=axes[1], bins=20,
                        color="#43A047", edgecolor="white")
axes[1].set_title("Chain length distribution (faults per cascade day)")
axes[1].set_xlabel("chain_length")

# Cascade days over time
if "transit_day" in df.columns:
    df.groupby("transit_day").size().rolling(7).mean().plot(
        ax=axes[2], color="#E53935", linewidth=0.7)
    axes[2].set_title("Cascade days per day (7-day rolling avg)")

plt.tight_layout(); plt.savefig("/tmp/ps2_eda.png", dpi=100); plt.show()

In [ ]:
# ── CELL 6 : Feature engineering & temporal splits ───────────────────────────
# PRIMARY target : has_cash_cascade  (BHU+CHU co-failure — most costly cascade type)
# Secondary targets stored separately; NOT included as features.
# All rows in this table are cascade days (chain_length >= 2) — no 0-class padding.
# chain_length is already a gold column — do NOT re-derive from subsystem_chain.
PRIMARY_TARGET    = "has_cash_cascade"
SECONDARY_TARGETS = ["has_gate_mech_in_chain","has_printer_in_chain",
                     "has_csc_reader_in_chain"]

EXCLUDE_COLS = (
    ["DEVICE_ID","DEVICE_KEY","transit_day","mars_device_category",
     "DEVICE_NAME","FACILITY_ID","FACILITY_NAME","OPERATOR_ID","OPERATOR_NAME",
     "DEVICE_SERIAL_NUMBER","DEVICE_CONTROL_GROUP_TYPE_NAME","BUS_ID",
     "subsystem_chain","event_type_chain","severity_chain",
     "chain_start_dtm","chain_end_dtm",
     "first_subsystem","last_subsystem",
     "ci_related_devices","shared_facility_chain","svn_ci_data_available",
     "has_cascade",          # always True in this table — degenerate
     PRIMARY_TARGET]
    + SECONDARY_TARGETS
)
FEATURE_COLS = [
    c for c in df.columns
    if c not in EXCLUDE_COLS
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
print(f"Features: {len(FEATURE_COLS)}  →  {FEATURE_COLS}")

train_df = df[df["transit_day"] <  "2025-10-01"]
val_df   = df[(df["transit_day"] >= "2025-10-01") & (df["transit_day"] < "2026-01-01")]
test_df  = df[df["transit_day"] >= "2026-01-01"]

X_train, y_train = train_df[FEATURE_COLS].values, train_df[PRIMARY_TARGET].values
X_val,   y_val   = val_df[FEATURE_COLS].values,   val_df[PRIMARY_TARGET].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df[PRIMARY_TARGET].values
X_cv             = np.vstack([X_train, X_val])
y_cv             = np.concatenate([y_train, y_val])

neg, pos         = (y_train==0).sum(), (y_train==1).sum()
SCALE_POS_WEIGHT = round(neg/pos, 2) if pos > 0 else 1.0
print(f"Train:{len(X_train):,}  Val:{len(X_val):,}  Test:{len(X_test):,}")
print(f"has_cash_cascade rate (train): {y_train.mean():.2%}")
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")

In [ ]:
# ── CELL 7 : Feature Store ────────────────────────────────────────────────────
fs_df = train_df[FEATURE_COLS + [PRIMARY_TARGET] + SECONDARY_TARGETS
                 + ["DEVICE_ID","transit_day"]].copy()
fs_df["event_time"] = fs_df["transit_day"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
fs_df["DEVICE_ID"]  = fs_df["DEVICE_ID"].astype(str)
fs_df               = fs_df.drop(columns=["transit_day"])

def dtype_to_fs(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

feature_group = FeatureGroup(
    name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
    feature_definitions=[
        FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
        for c in fs_df.columns
    ])
try:
    feature_group.create(
        s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name="DEVICE_ID",
        event_time_feature_name="event_time",
        role_arn=ROLE, enable_online_store=True,
        description="PS2 cascade chain features — Chicago Ventra")
    status = ""
    while status != "Created":
        time.sleep(15)
        status = feature_group.describe().get("FeatureGroupStatus")
        print(f"  {status}")
except Exception as e:
    print(f"Feature Group: {e}")

In [ ]:
# ── CELL 8 : Ingest to Feature Store ─────────────────────────────────────────
feature_group.ingest(data_frame=fs_df.reset_index(drop=True), max_workers=4, wait=True)
print(f"Ingested {len(fs_df):,} records → {FS_GROUP_NAME}")

In [ ]:
# ── CELL 9 : Helpers ──────────────────────────────────────────────────────────
COMMON_TAGS = {"ps":"PS2","target":PRIMARY_TARGET,"city":"chicago",
               "gold_version":str(GOLD_VERSION),"architecture":"CUBIC-MARS-AWS-v6"}

def evaluate(model, X, y, split="test"):
    prob = model.predict_proba(X)[:,1]
    pred = (prob>=0.5).astype(int)
    return {f"{split}_auc" :round(roc_auc_score(y,prob),4),
            f"{split}_ap"  :round(average_precision_score(y,prob),4),
            f"{split}_f1"  :round(f1_score(y,pred),4)}

def log_cm(model, X, y, name):
    pred = (model.predict_proba(X)[:,1]>=0.5).astype(int)
    cm   = confusion_matrix(y,pred)
    fig,ax=plt.subplots(figsize=(4,3))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                xticklabels=["No Cash Cascade","Cash Cascade"],
                yticklabels=["No Cash Cascade","Cash Cascade"])
    ax.set_title(f"{name}")
    fig.tight_layout(); fig.savefig(f"/tmp/cm_{name}.png",dpi=80); plt.close(fig)
    mlflow.log_artifact(f"/tmp/cm_{name}.png")

In [ ]:
# ── CELL 10 : MODEL 1 — XGBoost (primary cascade classifier) ─────────────────
XGB_PARAMS = dict(
    n_estimators=600, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
    gamma=1.0, scale_pos_weight=SCALE_POS_WEIGHT,
    eval_metric="auc", early_stopping_rounds=40,
    tree_method="hist", random_state=42, n_jobs=-1,
)
with mlflow.start_run(run_name="xgboost_cash_cascade") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"gradient_boosting_xgb"})
    xgb = XGBClassifier(**XGB_PARAMS)
    xgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],verbose=100)
    vm=evaluate(xgb,X_val,y_val,"val"); tm=evaluate(xgb,X_test,y_test,"test")
    mlflow.log_params({**XGB_PARAMS,"best_iter":xgb.best_iteration})
    mlflow.log_metrics({**vm,**tm}); log_cm(xgb,X_test,y_test,"xgb")
    sig=infer_signature(X_train,xgb.predict_proba(X_train))
    mlflow.sklearn.log_model(xgb,"model",signature=sig); XGB_RUN=run.info.run_id
print(f"XGB has_cash_cascade  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 11 : MODEL 2 — LightGBM ─────────────────────────────────────────────
with mlflow.start_run(run_name="lightgbm_cash_cascade") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"gradient_boosting_lgbm"})
    lgb = LGBMClassifier(
        n_estimators=600, num_leaves=63, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        scale_pos_weight=SCALE_POS_WEIGHT, n_jobs=-1, random_state=42, verbose=-1)
    lgb.fit(X_train,y_train,eval_set=[(X_val,y_val)],
            callbacks=[early_stopping(40,verbose=False),log_evaluation(100)])
    vm=evaluate(lgb,X_val,y_val,"val"); tm=evaluate(lgb,X_test,y_test,"test")
    mlflow.log_params({"n_estimators":600,"scale_pos_weight":SCALE_POS_WEIGHT,
                       "best_iter":lgb.best_iteration_})
    mlflow.log_metrics({**vm,**tm}); log_cm(lgb,X_test,y_test,"lgb")
    sig=infer_signature(X_train,lgb.predict_proba(X_train))
    mlflow.sklearn.log_model(lgb,"model",signature=sig); LGB_RUN=run.info.run_id
print(f"LGB has_cash_cascade  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 12 : Markov Chain — cascade transition probability matrix ────────────
# Gold table grain = cascade days only (chain_length >= 2); train_df is already
# all-cascades — no filter needed.
# subsystem_chain separator = '->' (gold array_join with '->')

from collections import Counter

if "subsystem_chain" in df.columns:
    # Parse chains: "BHU->CHU->PRINTER" → ["BHU","CHU","PRINTER"]
    chains = (train_df["subsystem_chain"]
              .dropna()
              .str.split("->")
              .apply(lambda c: [s.strip() for s in c])
              .tolist())

    # Build transition counts
    states    = sorted(set(s for chain in chains for s in chain if s))
    state_idx = {s:i for i,s in enumerate(states)}
    n         = len(states)
    trans     = np.zeros((n,n))
    for chain in chains:
        for a,b in zip(chain[:-1],chain[1:]):
            if a in state_idx and b in state_idx:
                trans[state_idx[a],state_idx[b]] += 1

    # Normalise rows → probability matrix
    row_sums = trans.sum(axis=1, keepdims=True)
    row_sums[row_sums==0] = 1
    trans_prob = trans / row_sums
    trans_df = pd.DataFrame(trans_prob, index=states, columns=states)

    # Visualise
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(trans_df, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
                linewidths=0.5)
    ax.set_title("PS2 Markov Chain — Subsystem Cascade Transition Probabilities\n"
                 "(train split; all rows are cascade days)")
    fig.tight_layout()
    fig.savefig("/tmp/markov_transition.png", dpi=100); plt.close(fig)
    trans_df.to_csv("/tmp/markov_transition.csv")

    with mlflow.start_run(run_name="markov_chain_analysis") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"markov_chain","stage":"analysis"})
        mlflow.log_param("n_states",  n)
        mlflow.log_param("n_chains",  len(chains))

        # Most common cascade paths
        path_counts = Counter(["->".join(c) for c in chains])
        top_paths = pd.DataFrame(path_counts.most_common(10),
                                  columns=["path","count"])
        top_paths.to_csv("/tmp/top_cascade_paths.csv", index=False)
        mlflow.log_artifact("/tmp/top_cascade_paths.csv")
        mlflow.log_artifact("/tmp/markov_transition.png")
        mlflow.log_artifact("/tmp/markov_transition.csv")

        # Log significant transitions as metrics
        for i,s1 in enumerate(states):
            for j,s2 in enumerate(states):
                if trans_prob[i,j] > 0.05:
                    mlflow.log_metric(f"trans_{s1}_to_{s2}", round(trans_prob[i,j],3))

        MARKOV_RUN = run.info.run_id

    print(f"\nTop cascade paths:\n{top_paths.head(5).to_string(index=False)}")
    print(f"\nHighest transition probabilities:")
    high_trans = [(states[i],states[j],trans_prob[i,j])
                  for i in range(n) for j in range(n)
                  if trans_prob[i,j]>0.15]
    high_trans.sort(key=lambda x:-x[2])
    for s1,s2,p in high_trans[:5]:
        print(f"  {s1} → {s2} : {p:.2%}")
else:
    print("subsystem_chain not in Gold — Markov analysis skipped")
    MARKOV_RUN = None

In [ ]:
# ── CELL 13 : HMM — 3-state Gaussian HMM on per-device cascade severity ───────
# Features: gold cascade-severity metrics (chain_length, critical_in_chain,
#            oos_in_chain, chain_span_min, distinct_subsystems).
# Grain is (device_id, transit_day) cascade days — sort by transit_day.
# States model severity progression: Minor → Moderate → Critical cascade.
# NOTE: all rows ARE cascade days (chain_length >= 2); HMM learns which cascade
# days cluster into low/medium/high severity regimes, not pre-vs-cascade.

HMM_FEATURES = [c for c in ["chain_length","critical_in_chain","oos_in_chain",
                              "chain_span_min","distinct_subsystems"]
                 if c in df.columns]

if len(HMM_FEATURES) >= 2:
    hmm_data = (train_df.sort_values(["DEVICE_ID","transit_day"])
                         .groupby("DEVICE_ID")[HMM_FEATURES]
                         .apply(lambda x: x.values.tolist())
                         .reset_index()
                         .rename(columns={0:"sequence"}))

    sequences = [np.array(seq) for seq in hmm_data["sequence"] if len(seq)>=3]
    if not sequences:
        print("No device has >= 3 cascade days in train split — HMM skipped")
        HMM_RUN = None
    else:
        X_hmm   = np.vstack(sequences)
        lengths = [len(s) for s in sequences]

        model_hmm = hmm.GaussianHMM(
            n_components=3, covariance_type="diag",
            n_iter=100, random_state=42, verbose=False)
        model_hmm.fit(X_hmm, lengths)

        with mlflow.start_run(run_name="hmm_3state") as run:
            mlflow.set_tags({**COMMON_TAGS,"model_type":"gaussian_hmm","n_states":"3"})
            mlflow.log_params({
                "n_components"   : 3,
                "covariance_type": "diag",
                "n_iter"         : 100,
                "hmm_features"   : str(HMM_FEATURES),
                "n_sequences"    : len(sequences),
            })
            for s in range(3):
                for fi, fn in enumerate(HMM_FEATURES):
                    mlflow.log_metric(f"state{s}_mean_{fn}",
                                       round(float(model_hmm.means_[s][fi]),4))

            trans_mat_df = pd.DataFrame(
                model_hmm.transmat_,
                columns=[f"→S{i}" for i in range(3)],
                index=[f"S{i}" for i in range(3)])
            trans_mat_df.to_csv("/tmp/hmm_transmat.csv")
            mlflow.log_artifact("/tmp/hmm_transmat.csv")

            joblib.dump(model_hmm, "/tmp/ps2_hmm.joblib")
            mlflow.log_artifact("/tmp/ps2_hmm.joblib","hmm_model")
            HMM_RUN = run.info.run_id

        print(f"\nHMM state means (cascade severity regime):")
        STATE_LABELS = ["Minor cascade","Moderate cascade","Critical cascade"]
        for s in range(3):
            print(f"  State {s} ({STATE_LABELS[s]}): "
                  f"{dict(zip(HMM_FEATURES,model_hmm.means_[s].round(3)))}")
        print(f"\nHMM transition matrix:\n{trans_mat_df.to_string()}")
else:
    print("Insufficient HMM features in Gold — skipped")
    HMM_RUN = None

In [ ]:
# ── CELL 14 : Association Rules — co-failing subsystem pairs ──────────────────
# Gold table grain = (device_id, transit_day) cascade days only.
# Each row's subsystem_chain IS the itemset for that cascade day.
# subsystem_chain separator = '->' (gold array_join with '->')
# Deduplicate items within each basket (a subsystem can appear > once if
# listed multiple times in a chain string).

if "subsystem_chain" in df.columns:
    basket = [
        list(set(s.strip() for s in row.split("->") if s.strip()))
        for row in train_df["subsystem_chain"].dropna()
    ]
    # Keep only baskets with >= 2 items (Association Rules require at least 2)
    basket = [b for b in basket if len(b) >= 2]

    if not basket:
        print("No multi-item baskets found — Association Rules skipped")
        RULES_RUN = None
    else:
        te     = TransactionEncoder()
        te_arr = te.fit(basket).transform(basket)
        basket_df = pd.DataFrame(te_arr, columns=te.columns_)

        try:
            frequent_sets = apriori(basket_df, min_support=0.05, use_colnames=True)
            rules = association_rules(frequent_sets, metric="lift", min_threshold=1.5)
            rules = rules.sort_values("lift", ascending=False)
            rules_out = rules[["antecedents","consequents","support","confidence","lift"]].copy()
            rules_out["antecedents"] = rules_out["antecedents"].apply(lambda x: ", ".join(sorted(x)))
            rules_out["consequents"] = rules_out["consequents"].apply(lambda x: ", ".join(sorted(x)))
            rules_out.to_csv("/tmp/association_rules.csv", index=False)

            with mlflow.start_run(run_name="association_rules") as run:
                mlflow.set_tags({**COMMON_TAGS,"model_type":"association_rules"})
                mlflow.log_params({
                    "min_support"    : 0.05,
                    "min_lift"       : 1.5,
                    "n_rules"        : len(rules),
                    "n_frequent_sets": len(frequent_sets),
                    "n_baskets"      : len(basket),
                })
                mlflow.log_artifact("/tmp/association_rules.csv")
                for i, row in rules_out.head(5).iterrows():
                    mlflow.log_metric(f"rule{i}_lift", round(float(row["lift"]),3))
                RULES_RUN = run.info.run_id

            print(f"\nTop 5 association rules (subsystem co-failure):")
            print(rules_out.head(5).to_string(index=False))
        except Exception as e:
            print(f"Association rules error: {e}"); RULES_RUN=None
else:
    print("subsystem_chain not available — association rules skipped")
    RULES_RUN = None

In [ ]:
# ── CELL 15 : Optuna — tune XGBoost (all-fold rule) ──────────────────────────
N_TRIALS = 50
TS_CV    = TimeSeriesSplit(n_splits=5)

def optuna_obj(trial):
    p = dict(
        n_estimators    =trial.suggest_int("n_estimators",200,800),
        max_depth       =trial.suggest_int("max_depth",4,9),
        learning_rate   =trial.suggest_float("learning_rate",0.01,0.2,log=True),
        subsample       =trial.suggest_float("subsample",0.5,1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree",0.5,1.0),
        min_child_weight=trial.suggest_int("min_child_weight",1,30),
        gamma           =trial.suggest_float("gamma",0.0,3.0),
        reg_alpha       =trial.suggest_float("reg_alpha",1e-4,10.,log=True),
        scale_pos_weight=SCALE_POS_WEIGHT,
        tree_method="hist", random_state=42, n_jobs=-1,
    )
    scores=[]
    for step,(tr,va) in enumerate(TS_CV.split(X_cv)):
        m=XGBClassifier(**p)
        m.fit(X_cv[tr],y_cv[tr],eval_set=[(X_cv[va],y_cv[va])],verbose=False)
        scores.append(roc_auc_score(y_cv[va],m.predict_proba(X_cv[va])[:,1]))
        trial.report(np.mean(scores),step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

study=optuna.create_study(direction="maximize",pruner=optuna.pruners.MedianPruner())
study.optimize(optuna_obj,n_trials=N_TRIALS,show_progress_bar=True)
BEST_PARAMS={**study.best_params,"scale_pos_weight":SCALE_POS_WEIGHT,
             "tree_method":"hist","random_state":42,"n_jobs":-1}

with mlflow.start_run(run_name="xgboost_optuna") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"xgboost_optuna_tuned"})
    mlflow.log_params({**BEST_PARAMS,"optuna_trials":N_TRIALS,"optuna_cv":round(study.best_value,4)})
    xgb_tuned=XGBClassifier(**BEST_PARAMS)
    xgb_tuned.fit(X_cv,y_cv,verbose=False)
    vm=evaluate(xgb_tuned,X_val,y_val,"val"); tm=evaluate(xgb_tuned,X_test,y_test,"test")
    mlflow.log_metrics({**vm,**tm}); log_cm(xgb_tuned,X_test,y_test,"xgb_optuna")
    sig=infer_signature(X_train,xgb_tuned.predict_proba(X_train))
    mlflow.sklearn.log_model(xgb_tuned,"model",signature=sig)
    XGB_TUNED_RUN=run.info.run_id
print(f"XGB(Optuna) val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 16 : Stacking Ensemble ───────────────────────────────────────────────
with mlflow.start_run(run_name="stacking_ensemble") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"stacking_ensemble"})
    mlflow.log_params({"base":"XGB+LGB+CatBoost+RF","final":"LR+isotonic",
                        "cv":"TimeSeriesSplit(5)","smote":False})
    base=[
        ("xgb",XGBClassifier(**{**BEST_PARAMS,"n_estimators":300},verbosity=0)),
        ("lgb",LGBMClassifier(n_estimators=300,learning_rate=0.05,
                               scale_pos_weight=SCALE_POS_WEIGHT,n_jobs=-1,
                               random_state=42,verbose=-1)),
        ("cb", CatBoostClassifier(iterations=300,learning_rate=0.05,
                                   scale_pos_weight=SCALE_POS_WEIGHT,
                                   random_seed=42,verbose=0)),
        ("rf", RandomForestClassifier(n_estimators=200,max_depth=10,
                                       class_weight="balanced_subsample",
                                       n_jobs=-1,random_state=42)),
    ]
    stack=StackingClassifier(
        estimators=base,
        final_estimator=CalibratedClassifierCV(
            LogisticRegression(C=1.0,max_iter=500),cv=3,method="isotonic"),
        cv=TimeSeriesSplit(n_splits=5),stack_method="predict_proba",
        passthrough=True,n_jobs=-1)
    stack.fit(X_cv,y_cv)
    vm=evaluate(stack,X_val,y_val,"val"); tm=evaluate(stack,X_test,y_test,"test")
    mlflow.log_metrics({**vm,**tm}); log_cm(stack,X_test,y_test,"stack")
    sig=infer_signature(X_train,stack.predict_proba(X_train))
    mlflow.sklearn.log_model(stack,"model",signature=sig)
    STACK_RUN=run.info.run_id
print(f"Stack  val_auc={vm['val_auc']}  test_auc={tm['test_auc']}")

In [ ]:
# ── CELL 17 : SHAP — xgb_tuned ───────────────────────────────────────────────
X_shap   = pd.DataFrame(X_test[:2000],columns=FEATURE_COLS)
explainer= shap.TreeExplainer(xgb_tuned)
sv       = explainer(X_shap)

with mlflow.start_run(run_name="shap_analysis") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"shap_analysis"})
    shap.plots.bar(sv,max_display=20,show=False)
    fig = plt.gcf(); ax = plt.gca()
    ax.set_title("PS2 XGBoost — SHAP Feature Importance",pad=12)
    fig.tight_layout(); fig.savefig("/tmp/ps2_shap_bar.png",dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/ps2_shap_bar.png")
    shap_imp=(pd.Series(np.abs(sv.values).mean(0),index=FEATURE_COLS)
                .sort_values(ascending=False).reset_index()
                .rename(columns={"index":"feature",0:"mean_abs_shap"}))
    shap_imp.to_csv("/tmp/ps2_shap_importance.csv",index=False)
    mlflow.log_artifact("/tmp/ps2_shap_importance.csv")
print(f"Top 5 SHAP:\n{shap_imp.head(5).to_string(index=False)}")

In [ ]:
# ── CELL 18 : Select champion + register (MLflow + SageMaker) ────────────────
all_runs=mlflow.search_runs(experiment_names=[EXPERIMENT_NAME],
                             order_by=["metrics.test_auc DESC"])
all_runs=all_runs[all_runs["tags.mlflow.runName"].str.contains(
    "xgboost|lightgbm|stack",na=False)]
CHAMPION_RUN_ID = all_runs.iloc[0]["run_id"]
CHAMPION_NAME   = all_runs.iloc[0]["tags.mlflow.runName"]
CHAMPION_AUC    = all_runs.iloc[0]["metrics.test_auc"]
print(f"Champion: {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}")

# MLflow registration
mlfc=mlflow.tracking.MlflowClient()
try:
    mlfc.create_registered_model(MLFLOW_MODEL_NAME,
        description="PS2 Cascade Chain Analysis — Chicago Ventra")
except: pass
mv=mlfc.create_model_version(
    name=MLFLOW_MODEL_NAME, source=f"runs:/{CHAMPION_RUN_ID}/model",
    run_id=CHAMPION_RUN_ID,
    tags={"algorithm":CHAMPION_NAME,"test_auc":str(CHAMPION_AUC),
          "gold_version":str(GOLD_VERSION)})
mlfc.transition_model_version_stage(MLFLOW_MODEL_NAME,mv.version,"Staging")
MLFLOW_VERSION=mv.version
print(f"MLflow: {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION} → Staging")

# SageMaker registration
champion_model=mlflow.sklearn.load_model(f"runs:/{CHAMPION_RUN_ID}/model")
joblib.dump(champion_model,"/tmp/ps2_champion.joblib")
joblib.dump(FEATURE_COLS,"/tmp/ps2_feature_cols.joblib")
test_metrics=evaluate(champion_model,X_test,y_test,"test")
S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,Key="sagemaker/ps2/metrics/test_metrics.json",
                     Body=json.dumps(test_metrics),ContentType="application/json")
with tarfile.open("/tmp/model.tar.gz","w:gz") as tar:
    tar.add("/tmp/ps2_champion.joblib",arcname="ps2_champion.joblib")
    tar.add("/tmp/ps2_feature_cols.joblib",arcname="ps2_feature_cols.joblib")
MODEL_S3_KEY=f"sagemaker/ps2/model-v{MLFLOW_VERSION}/model.tar.gz"
S3_CLIENT.upload_file("/tmp/model.tar.gz",ARTIFACT_BUCKET,MODEL_S3_KEY)

import sagemaker.image_uris as sm_images
image_uri=sm_images.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
    instance_type="ml.m5.large", image_scope="inference")
try:
    SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageGroupDescription="PS2 Cascade Analysis — Chicago Ventra")
except: pass
resp=SM_CLIENT.create_model_package(
    ModelPackageGroupName=MODEL_REGISTRY_NAME,
    ModelPackageDescription=f"{CHAMPION_NAME}|AUC={CHAMPION_AUC:.4f}|Gv{GOLD_VERSION}",
    InferenceSpecification={
        "Containers":[{"Image":image_uri,"ModelDataUrl":f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"}],
        "SupportedContentTypes":["text/csv"],"SupportedResponseMIMETypes":["application/json"],
        "SupportedRealtimeInferenceInstanceTypes":["ml.m5.large"]},
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"ps":"PS2","algorithm":CHAMPION_NAME,
        "test_auc":str(round(CHAMPION_AUC,4)),"mlflow_version":str(MLFLOW_VERSION)})
SM_MODEL_PKG_ARN=resp["ModelPackageArn"]
print(f"SM Package: {SM_MODEL_PKG_ARN}")

In [ ]:
# ── CELL 19 : Deploy + Monitors ──────────────────────────────────────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import (DefaultModelMonitor, ModelQualityMonitor,
                                      DataCaptureConfig, CronExpressionGenerator)
from sagemaker.model_monitor.dataset_format import DatasetFormat

SM_CLIENT.update_model_package(ModelPackageArn=SM_MODEL_PKG_ARN,
                                ModelApprovalStatus="Approved")
registered=ModelPackage(role=ROLE,model_package_arn=SM_MODEL_PKG_ARN,
                         sagemaker_session=SM_SESSION)
predictor=registered.deploy(
    initial_instance_count=1,instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=DataCaptureConfig(
        enable_capture=True,sampling_percentage=100,
        destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps2/data-capture/",
        capture_options=["Input","Output"]))
print(f"Endpoint: {ENDPOINT_NAME}")

# Data Quality Monitor
monitor=DefaultModelMonitor(role=ROLE,instance_count=1,
    instance_type="ml.m5.large",sagemaker_session=SM_SESSION)
pd.DataFrame(X_train,columns=FEATURE_COLS).to_csv("/tmp/ps2_baseline.csv",index=False)
S3_CLIENT.upload_file("/tmp/ps2_baseline.csv",ARTIFACT_BUCKET,
                      "sagemaker/ps2/baseline/baseline.csv")
monitor.suggest_baseline(
    baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps2/baseline/baseline.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps2/baseline/output/",
    wait=True,logs=False)
monitor.create_monitoring_schedule(
    monitor_schedule_name="chicago-ps2-data-quality",
    endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps2/monitor-output/",
    statistics=monitor.baseline_statistics(),constraints=monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics=True)

cw=boto3.client("cloudwatch",region_name=REGION)
cw.put_metric_alarm(AlarmName="PS2-DataQuality-ViolationRate",
    ComparisonOperator="GreaterThanThreshold",EvaluationPeriods=1,
    MetricName="feature_baseline_drift_percentage",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    Period=86400,Statistic="Average",Threshold=3.0,
    AlarmDescription="PS2 cascade input violation >3% — CUBIC MARS 97% gate",
    Dimensions=[{"Name":"Endpoint","Value":ENDPOINT_NAME}],
    TreatMissingData="notBreaching")
print("Monitors + CloudWatch alarm configured.")

In [ ]:
# ── CELL 20 : Summary ─────────────────────────────────────────────────────────
print("="*60)
print("PS2 — Cascade Chain Analysis — Complete")
print("="*60)
print(f"Gold table      : mars_dev.gold.device_ps2_chains")
print(f"                  (cascade days only — chain_length >= 2)")
print(f"Primary target  : {PRIMARY_TARGET}  (BHU+CHU bill/coin handler co-failure)")
print(f"Champion model  : {CHAMPION_NAME}  AUC={CHAMPION_AUC:.4f}")
print(f"MLflow model    : {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION}")
print(f"SM endpoint     : {ENDPOINT_NAME}")
print(f"Feature Store   : {FS_GROUP_NAME}")
print(f"Markov chain    : run={MARKOV_RUN}")
print(f"HMM (3 states)  : run={HMM_RUN}")
print(f"Association rules: run={RULES_RUN}")
print(f"\nFeasibility gap : 15% — SVN_STAGE.CMDB_CI/CMDB_REL = 0 rows")
print("  Impact: shared_infra_device_count + dependency_depth features blocked")
print("  Workaround: temporal co-location cascade detection (implemented)")